In [19]:
import pandas as pd
import re
from pathlib import Path
from urllib.parse import urlparse

In [20]:
input_file = Path(r"D:\Proyek FEB\lengkapin data\sinta_results.xlsx")
output_file = Path(r"sinta1_siap_mapping.xlsx")
df = pd.read_excel(input_file)
df.head()

,title,matched_title,sinta_year,scopus_q,quartile_text,scopus_citation,citation_text,final_link,detail_or_scopus_url,status,link_status,http_status,source_url,error
0,Ethnic identity and internal migration decisio...,Ethnic identity and internal migration decisio...,2020.0,Q1,Q1 Journal,24.0,24 cited,https://doi.org/10.1080/1369183X.2018.1561252,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
1,The role of service quality within Indonesian ...,The role of service quality within Indonesian ...,2020.0,Q2,Q2 Journal,60.0,60 cited,https://doi.org/10.1108/JIMA-03-2017-0033,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
2,Developing Islamic crowdfunding website platfo...,Developing Islamic crowdfunding website platfo...,2020.0,Q2,Q2 Journal,48.0,48 cited,https://doi.org/10.1108/JIMA-02-2019-0022,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
3,Exploration of pilgrimage tourism in Indonesia,Exploration of pilgrimage tourism in Indonesia,2020.0,Q2,Q2 Journal,33.0,33 cited,https://doi.org/10.1108/JIMA-10-2018-0188,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN
4,A critical assessment of retail sovereign suku...,A critical assessment of retail sovereign suku...,2020.0,Q2,Q2 Journal,9.0,9 cited,https://doi.org/10.1108/QRFM-10-2018-0109,https://www.scopus.com/record/display.uri?eid=...,found,existing_link_kept,200.0,https://sinta.kemdiktisaintek.go.id/scopus/?q=...,NaN


In [21]:
def clean_text(value):
    if pd.isna(value):
        return None
    
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    
    return value if value else None


def clean_year(value):
    if pd.isna(value):
        return None
    
    text = str(value).strip()
    
    match = re.search(r"(19|20)\d{2}", text)
    if match:
        return int(match.group(0))
    
    return None


def clean_quartile(value):
    if pd.isna(value):
        return None
    
    text = str(value).strip()
    
    if re.search(r"\bno-?Q\b", text, flags=re.IGNORECASE):
        return "no-Q"
    
    match = re.search(r"\bQ\s*([1-4])\b", text, flags=re.IGNORECASE)
    if match:
        return f"Q{match.group(1)}"
    
    return None


def clean_citation(value):
    if pd.isna(value):
        return None
    
    text = str(value).strip()
    
    match = re.search(r"(\d+)", text)
    if match:
        return int(match.group(1))
    
    return None


def clean_link_like_main_data(value):
    """
    Format disesuaikan dengan data utama:
    - Jika link berisi DOI, simpan sebagai DOI mentah: 10.xxxx/yyyy
    - Jika bukan DOI tapi URL artikel, simpan URL-nya
    - Jika kosong, None
    """
    if pd.isna(value):
        return None
    
    text = str(value).strip()
    
    if not text:
        return None
    
    # Ambil DOI dari berbagai bentuk:
    # 10.xxxx/...
    # https://doi.org/10.xxxx/...
    # http://dx.doi.org/10.xxxx/...
    doi_match = re.search(r"(10\.\d{4,9}/\S+)", text, flags=re.IGNORECASE)
    
    if doi_match:
        doi = doi_match.group(1)
        doi = doi.strip()
        doi = doi.rstrip(".,;)")
        return doi
    
    # Kalau bukan DOI tapi URL artikel, biarkan
    if text.startswith("http://") or text.startswith("https://"):
        return text
    
    return text

In [22]:
cleaned = pd.DataFrame()

cleaned["matched_title"] = df["matched_title"].apply(clean_text)
cleaned["sinta_year"] = df["sinta_year"].apply(clean_year)
cleaned["scopus_q"] = df["scopus_q"].apply(clean_quartile)
cleaned["scopus_citation"] = df["scopus_citation"].apply(clean_citation)
cleaned["final_link"] = df["final_link"].apply(clean_link_like_main_data)

cleaned.head()

,matched_title,sinta_year,scopus_q,scopus_citation,final_link
0,Ethnic identity and internal migration decisio...,2020.0,Q1,24.0,10.1080/1369183X.2018.1561252
1,The role of service quality within Indonesian ...,2020.0,Q2,60.0,10.1108/JIMA-03-2017-0033
2,Developing Islamic crowdfunding website platfo...,2020.0,Q2,48.0,10.1108/JIMA-02-2019-0022
3,Exploration of pilgrimage tourism in Indonesia,2020.0,Q2,33.0,10.1108/JIMA-10-2018-0188
4,A critical assessment of retail sovereign suku...,2020.0,Q2,9.0,10.1108/QRFM-10-2018-0109


In [23]:
cleaned.isna().sum()

matched_title      951
sinta_year         951
scopus_q           951
scopus_citation    951
final_link          89
dtype: int64

In [24]:
cleaned.to_excel(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: sinta1_siap_mapping.xlsx
